# ECE 285 - HW2: WikiArt Painting GAN

In [ ]:
!pip install -q pytorch-ignite kagglehub

import os
import torch

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/ECE285/hw2"
    os.makedirs(SAVE_DIR, exist_ok=True)
    os.chdir(SAVE_DIR)
except ImportError:
    SAVE_DIR = os.getcwd()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
import os

LOCAL_DATA_DIR = "/content/wikiart" if os.path.exists("/content") else os.path.abspath("./data/wikiart")
DATA_ROOT = os.path.join(LOCAL_DATA_DIR, "datasets", "steubk", "wikiart", "versions", "1")

if not os.path.isdir(DATA_ROOT) or not os.listdir(DATA_ROOT):
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    os.environ["KAGGLEHUB_CACHE"] = LOCAL_DATA_DIR
    import kagglehub
    DATA_ROOT = kagglehub.dataset_download("steubk/wikiart")

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def _scan_dir_for_images(directory, exts=IMAGE_EXTS):
    paths = []
    with os.scandir(directory) as entries:
        for entry in entries:
            if entry.is_dir(follow_symlinks=False):
                paths.extend(_scan_dir_for_images(entry.path, exts=exts))
            elif entry.name.lower().endswith(exts):
                paths.append(entry.path)
    return paths


def build_image_index(root_dir, style=None, exts=IMAGE_EXTS):
    if style is not None:
        style_path = os.path.join(root_dir, style)
        if not os.path.isdir(style_path):
            raise FileNotFoundError(f"Style folder not found: {style_path}")
        paths = _scan_dir_for_images(style_path, exts=exts)
        paths.sort()
        return paths

    style_dirs = sorted(
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    )

    image_paths = []
    if style_dirs:
        for style_name in style_dirs:
            style_path = os.path.join(root_dir, style_name)
            image_paths.extend(_scan_dir_for_images(style_path, exts=exts))
    else:
        image_paths.extend(_scan_dir_for_images(root_dir, exts=exts))

    image_paths.sort()
    return image_paths


style_dirs = sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
IMAGE_INDEX = build_image_index(DATA_ROOT)
print(f"DATA_ROOT = {DATA_ROOT}")
print(f"Found {len(style_dirs)} styles, {len(IMAGE_INDEX)} total images")

## Dataset & DataLoader

In [ ]:
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
import PIL.Image as Image
import PIL.ImageFile as ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import utils

from ignite.engine import Engine
from ignite.metrics import FID, InceptionScore

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None


class WikiArtDataset(Dataset):
    def __init__(self, root_dir, img_size=64, style=None, max_samples=None, seed=42, image_paths=None):
        self.img_size = img_size

        if image_paths is not None:
            if style is not None:
                style_prefix = os.path.normpath(os.path.join(root_dir, style)) + os.sep
                self.image_paths = [p for p in image_paths if os.path.normpath(p).startswith(style_prefix)]
            else:
                self.image_paths = list(image_paths)
        else:
            self.image_paths = build_image_index(root_dir, style=style)

        if max_samples is not None and max_samples < len(self.image_paths):
            rng = random.Random(seed)
            self.image_paths = rng.sample(self.image_paths, max_samples)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        try:
            img = Image.open(self.image_paths[idx])
            img.draft("RGB", (self.img_size * 2, self.img_size * 2))
            img = img.convert("RGB").resize((self.img_size, self.img_size), Image.BILINEAR)
            x = torch.from_numpy(np.array(img, dtype=np.float32)) / 127.5 - 1.0
            return x.permute(2, 0, 1)
        except Exception:
            return self.__getitem__((idx + 1) % len(self.image_paths))


def get_dataloaders(root_dir, img_size=64, batch_size=64, test_size=1024, style=None, max_samples=None, seed=42, image_paths=None):
    full_ds = WikiArtDataset(root_dir, img_size=img_size, style=style, max_samples=max_samples, seed=seed, image_paths=image_paths)

    num_total = len(full_ds)
    test_size = min(test_size, max(1, num_total // 10))
    train_size = num_total - test_size

    generator = torch.Generator().manual_seed(seed)
    train_ds, test_ds = torch.utils.data.random_split(full_ds, [train_size, test_size], generator=generator)

    print(f"Train: {len(train_ds)} | Test: {len(test_ds)} | Total: {num_total}")

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, persistent_workers=True, prefetch_factor=2, drop_last=True
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True, persistent_workers=True
    )
    return train_loader, test_loader

## GAN Architecture & Loss Function

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class Generator(nn.Module):
    def __init__(self, latent_dim=100, ngf=64, img_size=64, nc=3):
        super().__init__()
        assert img_size in (64, 128)
        self.latent_dim = latent_dim
        self.img_size = img_size

        layers = [
            nn.ConvTranspose2d(latent_dim, ngf * 8, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
        ]

        if img_size == 128:
            layers.extend([
                nn.ConvTranspose2d(ngf, ngf // 2, kernel_size=4, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(ngf // 2),
                nn.ReLU(True),
                nn.ConvTranspose2d(ngf // 2, nc, kernel_size=4, stride=2, padding=1, bias=False),
                nn.Tanh(),
            ])
        else:
            layers.extend([
                nn.ConvTranspose2d(ngf, nc, kernel_size=4, stride=2, padding=1, bias=False),
                nn.Tanh(),
            ])

        self.main = nn.Sequential(*layers)

    def forward(self, z):
        if z.dim() == 2:
            z = z.view(z.size(0), z.size(1), 1, 1)
        return self.main(z)


class Discriminator(nn.Module):
    def __init__(self, ndf=64, img_size=64, nc=3):
        super().__init__()
        assert img_size in (64, 128)
        self.img_size = img_size

        layers = []
        if img_size == 128:
            layers.extend([
                nn.Conv2d(nc, ndf // 2, kernel_size=4, stride=2, padding=1, bias=False),
                nn.LeakyReLU(0.2, inplace=True),
                nn.Conv2d(ndf // 2, ndf, kernel_size=4, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(ndf),
                nn.LeakyReLU(0.2, inplace=True),
            ])
        else:
            layers.extend([
                nn.Conv2d(nc, ndf, kernel_size=4, stride=2, padding=1, bias=False),
                nn.LeakyReLU(0.2, inplace=True),
            ])

        layers.extend([
            nn.Conv2d(ndf, ndf * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, kernel_size=4, stride=1, padding=0, bias=False),
            nn.Sigmoid(),
        ])

        self.main = nn.Sequential(*layers)

    def forward(self, x):
        return self.main(x).view(-1)


class GAN(nn.Module):
    def __init__(self, latent_dim=100, ngf=64, ndf=64, img_size=64, nc=3, device=DEVICE):
        super().__init__()
        self.latent_dim = latent_dim
        self.img_size = img_size
        self.device = device

        self.netG = Generator(latent_dim=latent_dim, ngf=ngf, img_size=img_size, nc=nc)
        self.netD = Discriminator(ndf=ndf, img_size=img_size, nc=nc)

        self.netG.apply(weights_init)
        self.netD.apply(weights_init)

    def generate(self, z):
        return self.netG(z)


gan_criterion = nn.BCELoss()

## Training, Plotting & Evaluation Functions

In [ ]:
def plot_loss_curves(steps, g_losses, d_losses, save_path="loss_curves.png", show=False):
    plt.figure(figsize=(7, 4))
    plt.plot(steps, g_losses, label="Generator Loss (G)")
    plt.plot(steps, d_losses, label="Discriminator Loss (D)")
    plt.xlabel("Training Step")
    plt.ylabel("BCE Loss")
    plt.title("GAN Training Loss Curves (Generator vs. Discriminator)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    if show:
        plt.show()
    plt.close()


def load_gan_checkpoint(model, checkpoint_path, optimizerG=None, optimizerD=None, device=DEVICE):
    ckpt = torch.load(checkpoint_path, map_location=device)
    if isinstance(ckpt, dict) and "netG_state_dict" in ckpt:
        model.netG.load_state_dict(ckpt["netG_state_dict"])
        model.netD.load_state_dict(ckpt["netD_state_dict"])
        if optimizerG is not None and "optimizerG_state_dict" in ckpt:
            optimizerG.load_state_dict(ckpt["optimizerG_state_dict"])
        if optimizerD is not None and "optimizerD_state_dict" in ckpt:
            optimizerD.load_state_dict(ckpt["optimizerD_state_dict"])
    elif isinstance(ckpt, dict):
        model.load_state_dict(ckpt)
    return ckpt


def train_gan(
    model,
    train_loader,
    test_loader=None,
    start_step=0,
    total_steps=10000,
    lr=2e-4,
    beta1=0.5,
    log_every_steps=100,
    eval_every_steps=1000,
    device=DEVICE,
    save_dir="./",
    resume_checkpoint=None,
):
    model.to(device)
    netG, netD = model.netG, model.netD

    optimizerG = torch.optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))
    optimizerD = torch.optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))

    steps_recorded, g_losses, d_losses = [], [], []
    fixed_noise = torch.randn(5, model.latent_dim, 1, 1, device=device)

    if resume_checkpoint and os.path.exists(resume_checkpoint):
        ckpt = load_gan_checkpoint(model, resume_checkpoint, optimizerG, optimizerD, device=device)
        if isinstance(ckpt, dict):
            if start_step == 0 and "step" in ckpt:
                start_step = ckpt["step"]
            steps_recorded = list(ckpt.get("steps", []))
            g_losses = list(ckpt.get("g_losses", []))
            d_losses = list(ckpt.get("d_losses", []))
            if "fixed_noise" in ckpt:
                fixed_noise = ckpt["fixed_noise"].to(device)
        print(f"Loaded checkpoint: {resume_checkpoint} (resuming at step {start_step + 1})")

    os.makedirs(save_dir, exist_ok=True)
    loss_plot_path = os.path.join(save_dir, "loss_curves.png")
    eval_log_path = os.path.join(save_dir, "test_eval_metrics.txt")

    real_label = 1.0
    fake_label = 0.0

    netG.train()
    netD.train()

    running_g_loss = 0.0
    running_d_loss = 0.0
    running_d_x = 0.0
    running_d_gz = 0.0
    window_steps = 0

    avg_g_loss = 0.0
    avg_d_loss = 0.0
    avg_d_x = 0.0
    avg_d_gz = 0.0

    train_iter = iter(train_loader)
    pbar = tqdm(range(start_step + 1, total_steps + 1), desc="Training Steps")

    for step in pbar:
        try:
            real_imgs = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            real_imgs = next(train_iter)

        real_imgs = real_imgs.to(device, non_blocking=True)
        b_size = real_imgs.size(0)

        netD.zero_grad()
        label_real = torch.full((b_size,), real_label, dtype=torch.float32, device=device)
        output_real = netD(real_imgs)
        errD_real = gan_criterion(output_real, label_real)
        errD_real.backward()
        D_x = output_real.mean().item()

        noise = torch.randn(b_size, model.latent_dim, 1, 1, device=device)
        fake_imgs = netG(noise)
        label_fake = torch.full((b_size,), fake_label, dtype=torch.float32, device=device)
        output_fake = netD(fake_imgs.detach())
        errD_fake = gan_criterion(output_fake, label_fake)
        errD_fake.backward()

        errD = errD_real + errD_fake
        optimizerD.step()

        netG.zero_grad()
        output_gen = netD(fake_imgs)
        errG = gan_criterion(output_gen, label_real)
        errG.backward()
        D_G_z2 = output_gen.mean().item()
        optimizerG.step()

        running_g_loss += errG.item()
        running_d_loss += errD.item()
        running_d_x += D_x
        running_d_gz += D_G_z2
        window_steps += 1

        pbar.set_postfix(G_loss=f"{errG.item():.4f}", D_loss=f"{errD.item():.4f}")

        if (log_every_steps and step % log_every_steps == 0) or step == total_steps:
            avg_g_loss = running_g_loss / max(1, window_steps)
            avg_d_loss = running_d_loss / max(1, window_steps)
            avg_d_x = running_d_x / max(1, window_steps)
            avg_d_gz = running_d_gz / max(1, window_steps)

            steps_recorded.append(step)
            g_losses.append(avg_g_loss)
            d_losses.append(avg_d_loss)

            running_g_loss = 0.0
            running_d_loss = 0.0
            running_d_x = 0.0
            running_d_gz = 0.0
            window_steps = 0

            print(
                f"Step [{step}/{total_steps}] | G Loss: {avg_g_loss:.4f} | D Loss: {avg_d_loss:.4f} | "
                f"D(x): {avg_d_x:.4f} | D(G(z)): {avg_d_gz:.4f}"
            )
            plot_loss_curves(steps_recorded, g_losses, d_losses, save_path=loss_plot_path, show=(step == total_steps))

        if eval_every_steps and (step % eval_every_steps == 0 or step == total_steps):
            ckpt_dict = {
                "step": step,
                "netG_state_dict": netG.state_dict(),
                "netD_state_dict": netD.state_dict(),
                "optimizerG_state_dict": optimizerG.state_dict(),
                "optimizerD_state_dict": optimizerD.state_dict(),
                "steps": steps_recorded,
                "g_losses": g_losses,
                "d_losses": d_losses,
                "fixed_noise": fixed_noise.cpu(),
            }
            ckpt_path = os.path.join(save_dir, f"gan_wikiart_step_{step}.pt")
            latest_path = os.path.join(save_dir, "gan_wikiart.pt")
            torch.save(ckpt_dict, ckpt_path)
            torch.save(ckpt_dict, latest_path)

            samples_path = os.path.join(save_dir, f"generated_samples_step_{step}.png")
            generate_images(model, num_images=5, device=device, save_path=samples_path, fixed_noise=fixed_noise)

            if test_loader is not None:
                incep_score, fid = compute_fid_and_is(model, test_loader, device=device)
                log_line = (
                    f"Step {step:06d}/{total_steps:06d} | LR: {lr:.2e} | "
                    f"G Loss: {avg_g_loss:.6f} | D Loss: {avg_d_loss:.6f} | "
                    f"D(x): {avg_d_x:.4f} | D(G(z)): {avg_d_gz:.4f} | "
                    f"IS: {incep_score:.4f} | FID: {fid:.4f}\n"
                )
                with open(eval_log_path, "a") as f:
                    f.write(log_line)

            netG.train()
            netD.train()

    return steps_recorded, g_losses, d_losses


def interpolate(batch):
    batch = (batch + 1.0) / 2.0
    out = F.interpolate(batch, size=(299, 299), mode="bicubic", align_corners=False)
    return out.clamp(0.0, 1.0)


def compute_fid_and_is(model, test_loader, device=DEVICE):
    model.to(device)
    model.eval()

    def evaluation_step(engine, batch):
        with torch.no_grad():
            z = torch.randn(batch.size(0), model.latent_dim, 1, 1, device=device)
            generated = model.generate(z).cpu()
            fake = interpolate(generated)
            real = interpolate(batch.cpu())
            return fake, real

    fid_metric = FID(device=device)
    is_metric = InceptionScore(device=device, output_transform=lambda x: x[0])

    evaluator = Engine(evaluation_step)
    fid_metric.attach(evaluator, "fid")
    is_metric.attach(evaluator, "is")

    evaluator.run(test_loader)
    metrics = evaluator.state.metrics
    print(f"IS: {metrics['is']:.4f} | FID: {metrics['fid']:.4f}")
    return metrics["is"], metrics["fid"]


def generate_images(model, num_images=10, device=DEVICE, save_path="generated_samples.png", fixed_noise=None):
    model.to(device)
    model.eval()
    with torch.no_grad():
        if fixed_noise is not None and len(fixed_noise) >= num_images:
            z = fixed_noise[:num_images].to(device)
        else:
            z = torch.randn(num_images, model.latent_dim, 1, 1, device=device)
        samples = model.generate(z).cpu()
        samples = ((samples + 1.0) / 2.0).clamp(0.0, 1.0)

    nrow = min(num_images, 5)
    num_rows = (num_images + nrow - 1) // nrow
    grid = utils.make_grid(samples, nrow=nrow, padding=2)
    plt.figure(figsize=(2.2 * nrow, 2.3 * num_rows))
    plt.imshow(grid.permute(1, 2, 0).numpy())
    plt.axis("off")
    plt.title(f"Generated WikiArt Painting Images ({num_images} samples)")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()
    plt.close()

## Run GAN Training & Evaluation

In [ ]:
HYPERPARAMS = {
    "root_dir": DATA_ROOT,
    "save_dir": SAVE_DIR,
    "style": "Impressionism",
    "max_samples": None,
    "img_size": 64,
    "batch_size": 64,
    "latent_dim": 128,
    "ngf": 64,
    "ndf": 64,
    "lr": 2e-4,
    "beta1": 0.5,
    "start_step": 0,
    "total_steps": 50000,
    "log_every_steps": 500,
    "eval_every_steps": 5000,
    "test_size": 1024,
    "resume_checkpoint": os.path.join(SAVE_DIR, "gan_wikiart.pt"),
}

train_loader, test_loader = get_dataloaders(
    root_dir=HYPERPARAMS["root_dir"],
    img_size=HYPERPARAMS["img_size"],
    batch_size=HYPERPARAMS["batch_size"],
    test_size=HYPERPARAMS["test_size"],
    style=HYPERPARAMS["style"],
    max_samples=HYPERPARAMS["max_samples"],
    image_paths=IMAGE_INDEX if ("IMAGE_INDEX" in globals() and HYPERPARAMS["root_dir"] == DATA_ROOT) else None,
)

model = GAN(
    latent_dim=HYPERPARAMS["latent_dim"],
    ngf=HYPERPARAMS["ngf"],
    ndf=HYPERPARAMS["ndf"],
    img_size=HYPERPARAMS["img_size"],
    device=DEVICE,
).to(DEVICE)

steps_recorded, g_losses, d_losses = train_gan(
    model,
    train_loader,
    test_loader=test_loader,
    start_step=HYPERPARAMS["start_step"],
    total_steps=HYPERPARAMS["total_steps"],
    lr=HYPERPARAMS["lr"],
    beta1=HYPERPARAMS["beta1"],
    log_every_steps=HYPERPARAMS["log_every_steps"],
    eval_every_steps=HYPERPARAMS["eval_every_steps"],
    resume_checkpoint=HYPERPARAMS["resume_checkpoint"],
    device=DEVICE,
    save_dir=HYPERPARAMS["save_dir"],
)

samples_path = os.path.join(HYPERPARAMS["save_dir"], "generated_samples.png")
generate_images(model, num_images=10, device=DEVICE, save_path=samples_path)
compute_fid_and_is(model, test_loader, device=DEVICE)